# Exercise 17: VLA Benchmark Design

This notebook builds a reproducible benchmark manifest with language paraphrases, distractors, and failure labels.

In [ ]:
import pandas as pd
from itertools import product

objects = ["cup", "block"]
colors = ["red", "blue"]
containers = ["bowl", "tray"]
templates = [
    "Place the {color} {object} in the {container}.",
    "Move the {color} {object} into the {container}.",
    "The {color} {object} should end up inside the {container}.",
]

rows = []
task_id = 0
for color, obj, container in product(colors, objects, containers):
    for template_id, template in enumerate(templates):
        rows.append({
            "task_id": task_id,
            "semantic_task": f"{color}_{obj}_to_{container}",
            "template_id": template_id,
            "instruction": template.format(color=color, object=obj, container=container),
            "target_object": f"{color} {obj}",
            "target_container": container,
            "distractor_required": True,
        })
    task_id += 1

manifest = pd.DataFrame(rows)
manifest.head(10)

In [ ]:
# Hold out one paraphrase template from training.
manifest["split"] = np.where(manifest["template_id"] == 2, "paraphrase_test", "train")
manifest.groupby(["split", "template_id"]).size()

In [ ]:
failure_taxonomy = pd.DataFrame({
    "failure_code": [
        "LANGUAGE_MISUNDERSTANDING",
        "WRONG_OBJECT",
        "SPATIAL_RELATION_ERROR",
        "GRASP_FAILURE",
        "CONTROL_FAILURE",
        "LATENCY_FAILURE",
        "SAFETY_VIOLATION",
        "REFUSAL_FAILURE",
        "RECOVERY_FAILURE",
    ],
    "description": [
        "Instruction meaning is interpreted incorrectly.",
        "Policy acts on the wrong object.",
        "Target relation or container is incorrect.",
        "Correct object is selected but not grasped.",
        "Action execution fails despite correct grounding.",
        "Inference delay destabilizes or interrupts execution.",
        "A safety constraint is violated.",
        "Policy acts when it should refuse an infeasible request.",
        "Policy cannot recover after an intermediate failure.",
    ],
})
failure_taxonomy

## Required report

Report closed-loop task success separately for:

- seen templates;
- unseen paraphrases;
- unseen objects;
- distractor shifts;
- viewpoint shifts;
- infeasible-instruction refusal;
- target-robot or embodiment shifts.